In [1]:
# Firedrake coupled Saarelma–Connor solver tests
#
# Run from `tests/` (so `ROOT` points at the repo). Requires Firedrake +
# parent-class deps (OpenFUSIONToolkit, eqdsk/p-file inputs).

import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import os

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))


INPUT_DIR = Path("/Users/nelsonlab/codes/sc_inputs/inputs/PT_Hmode")
MHD_FP = Path('/Users/nelsonlab/codes/sc_inputs/saarelma-connor-inputs/CAKEgeqdsks/g158091.01935') # filepath to MHD paramter file
KPROF_FP = Path('/Users/nelsonlab/codes/sc_inputs/saarelma-connor-inputs/CAKEpfiles/p158091.01935')

# `solver-firedrake.py` has a hyphen — load by file path.
# from src.solver import saarelma_connor
from src.solver_nondim import saarelma_connor_nondim


firedrake:WARNING OMP_NUM_THREADS is not set or is set to a value greater than 1, we suggest setting OMP_NUM_THREADS=1 to improve performance


In [2]:
# Shared solve kwargs (override per cell as needed).
initial_guess = "tanh"
ne_inner_bc = "dirichlet"
SOLVE_KW = dict(
    x_res=20,
    fe_degree=2,
    initial_guess=initial_guess,
    ne_inner_bc=ne_inner_bc,   # Saarelma A7 default; see dirichlet comparison below
    linear_solver="lu",      # or "gamg" for GMRES + algebraic multigrid on J
    verbose=False,
)

In [2]:
# Build the model once — the expensive flux-surface-averaging and equilibrium
# loading runs here only, not inside the loop.

# Scan size (each dimension has N points)
N = 5

# Scan parameters
alpha_crits = np.logspace(-1, 1, N)
C_KBMs = np.logspace(-1, 1, N)
De_chie_etgs = np.logspace(-1, 1, N)
nFC_x0s = np.logspace(14, 18, N)
ncx_x0_ratios = np.logspace(0.1,1.25,N)

# Static parameters
P_tot_e = 5e6 # W, total heating power given to electrons (can be assumed to be half the total heating power according to S. Saarelma et al 2023 Nucl. Fusion 63 052002), will be read from TokTox

# Output data and files
success_fp = 'success_PTHmode.txt'
failure_fp = 'failure_PTHmode.txt'
error_messages_fp = 'error_messages_PTHmode.txt'
with open(success_fp, 'w') as f:
        f.write(f"alpha_crit, C_KBM, De_chie_etg, nFC_x0, ncx_x0_ratio, psi_N_inner\n")
with open(failure_fp, 'w') as f:
        f.write(f"alpha_crit, C_KBM, De_chie_etg, nFC_x0, ncx_x0_ratio, psi_N_inner\n")
with open(error_messages_fp, 'w') as f:
        f.write(f"alpha_crit, C_KBM, De_chie_etg, nFC_x0, ncx_x0_ratio, psi_N_inner, message\n")
ne_success_fp = 'compare_nondim'
verbose = False

base_model = saarelma_connor_nondim(
        P_tot_e      = P_tot_e,
        alpha_crit   = round(float(alpha_crits[0]), 3),
        C_KBM        = round(float(C_KBMs[0]), 3),
        De_chie_etg  = round(float(De_chie_etgs[0]), 3),
        nFC_x0       = round(float(nFC_x0s[0]), 3),
        ncx_x0_ratio = round(float(ncx_x0_ratios[0]), 3),
        mhd_fp       = MHD_FP,
        kprof_fp     = KPROF_FP,
        verbose      = verbose,
        # psi_N_inner_boundary = 0.85, # set to None to use adaptive inner boundary method
)
print("Base model built.")

Base model built.


In [6]:
# KBM Picard self-consistency test across 10 equilibria.
# ----------------------------------------------------------------------
# For each of 10 (g-file, p-file) pairs taken from the CAKE input
# directories (selection mirrors tests/multi_equilibria/param_err.ipynb
# and the equilibria cell in firedrake-tests_fparam_scan_nondim.ipynb):
#   - build a fresh saarelma_connor_nondim,
#   - solve with kbm_picard=False (gate frozen at the initial guess),
#   - solve with kbm_picard=True  (gate iterated to self-consistency),
#   - sample n_e at psi_N = 0.85 from each.
# Then plot:
#   (1) full n_e(x) profiles for all 10 equilibria, solid = no-Picard,
#       dashed = Picard, same colour per equilibrium.
#   (2) scatter of ne_noPicard(psi_N=0.85) vs ne_Picard(psi_N=0.85),
#       one point per equilibrium, with the y = x reference line.
# Picard-loop diagnostics (iter count, flickers, final ne change) are
# also printed for each equilibrium.

from collections import defaultdict
from scipy.interpolate import interp1d


# --- 1. Pick 10 equilibria (same algorithm as param_err.ipynb) --------------
GEQDSK_DIR = Path("/Users/nelsonlab/codes/sc_inputs/saarelma-connor-inputs/CAKEgeqdsks")
PFILE_DIR  = Path("/Users/nelsonlab/codes/sc_inputs/saarelma-connor-inputs/CAKEpfiles")


def initialize_inputs(equil_num, geqdsk_dir=GEQDSK_DIR, pfile_dir=PFILE_DIR):
    """Select equil_num g/p file pairs from the CAKE input directories.

    Prioritises one equilibrium per shot before adding additional
    times from already-seen shots.  Mirrors
    tests/multi_equilibria/param_err.ipynb.
    """
    geqdsk_dir = Path(geqdsk_dir); pfile_dir = Path(pfile_dir)
    g_by_suffix = {f.name[1:]: f for f in geqdsk_dir.glob("g*") if f.is_file()}
    p_by_suffix = {f.name[1:]: f for f in pfile_dir.glob("p*")  if f.is_file()}
    shared = sorted(set(g_by_suffix) & set(p_by_suffix))

    by_shot = defaultdict(list)
    for suffix in shared:
        shot, time = suffix.split(".", 1)
        by_shot[shot].append((time, suffix))
    for shot in by_shot:
        by_shot[shot].sort()

    shots = sorted(by_shot)
    selected, time_idx = [], 0
    while len(selected) < equil_num:
        added = False
        for shot in shots:
            if len(selected) >= equil_num:
                break
            entries = by_shot[shot]
            if time_idx < len(entries):
                suf = entries[time_idx][1]
                selected.append((str(g_by_suffix[suf]), str(p_by_suffix[suf])))
                added = True
        if not added:
            break
        time_idx += 1

    if len(selected) < equil_num:
        raise ValueError(
            f"Requested {equil_num} equilibria but only found {len(selected)} "
            f"matching g/p pairs in {geqdsk_dir} and {pfile_dir}"
        )
    return selected


equil_num    = 10
psi_N_target = 0.85
psi_N_inner  = 0.85

# Free parameters held fixed across all equilibria.  alpha_crit is set
# near the middle of the scan so the gate is "active" -- this is where
# the Picard loop has the biggest chance of moving the answer.
ac_fix = round(float(alpha_crits[2]),    3)
ck_fix = round(float(C_KBMs[2]),         3)
de_fix = round(float(De_chie_etgs[2]),   3)
nf_fix = round(float(nFC_x0s[2]),        3)
nc_fix = round(float(ncx_x0_ratios[2]),  3)

# Shared solve kwargs for both runs.  Use neumann BC + tanh initial
# guess: matches the most-recently exercised path.
COMMON_KW = dict(
    x_res=20,
    fe_degree=2,
    initial_guess="tanh",
    ne_inner_bc="neumann",
    bc_origin="p-file",
    linear_solver="lu",
    verbose=False,
)
PICARD_KW = dict(
    kbm_picard=True,
    kbm_picard_max_iters=20,
    kbm_picard_tol=1e-3,
    kbm_picard_relax=1.0,
    kbm_picard_auto_relax=True,
)

equilibria = initialize_inputs(equil_num)
print(f"Selected {len(equilibria)} equilibria; for each one we solve "
      f"once with kbm_picard=False and once with kbm_picard=True.\n")


# --- 2. helper: solve once and return ne(psi_N_target) ----------------------
def _solve_and_sample(model, picard_kw):
    """Solve, return (ne_at_psi_target, x_sol, ne_sol, picard_info)."""
    solve_kw = dict(COMMON_KW)
    if picard_kw:
        solve_kw.update(picard_kw)
    x_sol, ne_sol, _, _ = model.solve_coupled_nondim(**solve_kw)
    psi_of_x = interp1d(
        model.x_init, model.psi_N_pres,
        kind="linear", bounds_error=False, fill_value="extrapolate",
    )
    psi_sol = psi_of_x(x_sol)
    order = np.argsort(psi_sol)
    ne_at = float(np.interp(psi_N_target, psi_sol[order], ne_sol[order]))
    info  = getattr(model, "kbm_picard_info", None)
    return ne_at, np.asarray(x_sol, dtype=float), np.asarray(ne_sol, dtype=float), info


# --- 3. run scan over equilibria --------------------------------------------
ne_off_pts    = []
ne_on_pts     = []
eq_tags       = []
profiles_off  = []
profiles_on   = []
picard_infos  = []

for eq_idx, (mhd_fp, kprof_fp) in enumerate(equilibria):
    eq_tag = Path(mhd_fp).name[1:]
    print(f"[{eq_idx+1}/{len(equilibria)}] {eq_tag} ", end="", flush=True)

    try:
        model = saarelma_connor_nondim(
            P_tot_e      = P_tot_e,
            alpha_crit   = ac_fix,
            C_KBM        = ck_fix,
            De_chie_etg  = de_fix,
            nFC_x0       = nf_fix,
            ncx_x0_ratio = nc_fix,
            mhd_fp       = mhd_fp,
            kprof_fp     = kprof_fp,
            verbose      = False,
        )
        model.update_free_params(psi_N_inner_boundary=psi_N_inner)

        ne_off, x_off, prof_off, _              = _solve_and_sample(model, None)
        ne_on,  x_on,  prof_on,  info_on        = _solve_and_sample(model, PICARD_KW)
    except Exception as e:
        print(f"FAIL: {e}")
        continue

    ne_off_pts.append(ne_off)
    ne_on_pts.append(ne_on)
    eq_tags.append(eq_tag)
    profiles_off.append((x_off, prof_off))
    profiles_on.append((x_on, prof_on))
    picard_infos.append(info_on)

    rel_err = abs(ne_on - ne_off) / max(ne_off, 1.0)
    gate_init = info_on["gate_history"][0]
    gate_fin  = info_on["gate_history"][-1]
    print(
        f"ok  ne_noPic={ne_off:.3e}  ne_pic={ne_on:.3e}  "
        f"(diff = {100*rel_err:.2f}%, picard_iters={info_on['n_iters']}, "
        f"converged={info_on['converged']}, flickers={info_on['flicker_count']}, "
        f"gate {gate_init}->{gate_fin})"
    )
    del model

ne_off_pts = np.asarray(ne_off_pts)
ne_on_pts  = np.asarray(ne_on_pts)


# --- 4. plot ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), constrained_layout=True)

# Panel 1: full ne profiles per equilibrium
ax = axes[0]
cmap = plt.cm.viridis(np.linspace(0.1, 0.95, max(len(eq_tags), 1)))
for i, ((x_o, prof_o), (x_p, prof_p), tag) in enumerate(
    zip(profiles_off, profiles_on, eq_tags)
):
    ax.plot(x_o, prof_o / 1e19, "-",  color=cmap[i], lw=1.4, label=tag)
    ax.plot(x_p, prof_p / 1e19, "--", color=cmap[i], lw=1.4)
ax.set_xlabel(r"$x$ (m)")
ax.set_ylabel(r"$n_e$  ($10^{19}$ m$^{-3}$)")
ax.set_title("Solid = no Picard (gate frozen), dashed = with Picard (gate self-consistent)")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left", fontsize=7, ncol=2)

# Panel 2: ne(psi_N=psi_N_target) scatter
ax = axes[1]
sc = ax.scatter(
    ne_off_pts / 1e19, ne_on_pts / 1e19,
    c=np.arange(len(eq_tags)), cmap="viridis",
    s=80, edgecolor="k", zorder=3,
)
if len(ne_off_pts):
    lo = min(ne_off_pts.min(), ne_on_pts.min()) / 1e19
    hi = max(ne_off_pts.max(), ne_on_pts.max()) / 1e19
    pad = 0.05 * (hi - lo) if hi > lo else 0.05 * max(hi, 1.0)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1.0, label="y = x")
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)

ax.set_xlabel(rf"$n_e(\psi_N={psi_N_target})$ without Picard  ($10^{{19}}$ m$^{{-3}}$)")
ax.set_ylabel(rf"$n_e(\psi_N={psi_N_target})$ with Picard     ($10^{{19}}$ m$^{{-3}}$)")
ax.set_title(f"KBM Picard vs no-Picard at $\\psi_N={psi_N_target}$  (10 equilibria)")
ax.grid(True, alpha=0.3)
cb = plt.colorbar(sc, ax=ax, ticks=range(len(eq_tags)))
cb.set_label("equilibrium index")
cb.ax.set_yticklabels(eq_tags, fontsize=7)
ax.legend(loc="lower right")

plt.show()
# the valid range of psi_N_inner_boundary from the slope-zero inner limit to
# the nFC/nCX-threshold outer limit, then sample n_psi_inner_pts boundaries.
# n_iter = 0
i=0
for alpha_crit in alpha_crits:
        j=0
        for C_KBM in C_KBMs:
                k=0
                for De_chie_etg in De_chie_etgs:
                        l=0
                        for nFC_x0 in nFC_x0s:
                                for ncx_x0_ratio in ncx_x0_ratios:
                                        ac = round(float(alpha_crit), 3)
                                        ck = round(float(C_KBM), 3)
                                        de = round(float(De_chie_etg), 3)
                                        nf = round(float(nFC_x0), 3)
                                        nc = round(float(ncx_x0_ratio),3)

                                        # Step 1: apply free params with adaptive thresholds active so
                                        # inner_boundary_limits can compute the outer limit correctly.
                                        base_model.update_free_params(
                                                alpha_crit    = ac,
                                                C_KBM         = ck,
                                                De_chie_etg   = de,
                                                nFC_x0        = nf,
                                                ncx_x0_ratio  = nc,
                                        )

                                        # psi_scan = np.linspace(0.87, 0.94, N)
                                        psi_scan = [0.85]

                                        m=0
                                        for psi_b in psi_scan:
                                                psi_val = round(float(psi_b), 4)
                                                try:
                                                        base_model.update_free_params(
                                                                alpha_crit            = ac,
                                                                C_KBM                 = ck,
                                                                De_chie_etg           = de,
                                                                nFC_x0                = nf,
                                                                psi_N_inner_boundary  = psi_val,
                                                                ncx_x0_ratio  = nc,
                                                        )
                                                        x_sol, ne_sol, nFC_sol, nCX_sol = base_model.solve_coupled(**SOLVE_KW)
                                                        sol = {'x': x_sol, 'y': ne_sol, 'nFC': nFC_sol, 'nCX': nCX_sol}
                                                except Exception as e: # run fails
                                                        with open(failure_fp, 'a') as f:
                                                                f.write(f"{ac}, {ck}, {de}, {nf}, {nc}, {psi_val:.4f}\n")
                                                        with open(error_messages_fp, 'a') as f:
                                                                f.write(f"{ac}, {ck}, {de}, {nf}, {nc}, {psi_val:.4f}, {e}\n")
                                                        m+=1
                                                else: # run works
                                                        with open(success_fp, 'a') as f:
                                                                f.write(f"{ac}, {ck}, {de}, {nf}, {nc}, {psi_val:.4f}\n")
                                                        np.save(f'{ne_success_fp}/ne_a{ac}_C{ck}_D{de}_n{nf}_nc{nc}_b{psi_val:.4f}', sol, allow_pickle=True)
                                                        m+=1

                                                # n_iter += 1
                                                # if n_iter % 10 == 0: # RAM tracker
                                                #         gc.collect()
                                                #         print(f"  [{n_iter} iters] peak RAM ~ {_ram_mb():.0f} MB")
                                        l+=1
                                        # print(f"Completed {l} of {len(nFC_x0s)} nFC_x0s  [psi_N range: {psi_scan[0]:.3f}..{psi_scan[-1]:.3f}]") # progress logging
                        k+=1
                        # print(f"Completed {k} of {len(De_chie_etgs)} De_chie_etgs") # progress logging
                j+=1
                # print(f"Completed {j} of {len(C_KBMs)} C_KBMs") # progress logging
        i+=1
        print(f"Completed {i} of {len(alpha_crits)} alpha_crits") # progress logging

Completed 1 of 5 alpha_crits
Completed 2 of 5 alpha_crits
Completed 3 of 5 alpha_crits
Completed 4 of 5 alpha_crits
Completed 5 of 5 alpha_crits
